# Reproduce the paper: *When do relevance weights learn?* (ESANN 2027, submitted)

Part 1 builds Table 1, Fig. 1 and Fig. 2 from the aggregated results in `results/public/` and prints the numbers
quoted in the text (a few seconds, no model fitting). Part 2 re-fits one single run and shows how to start the full
re-run. Figures are written to `output/` in their final size (12.2 cm wide, 8 pt), as PDF and PNG.

Where each number of the paper comes from: `docs/numbers.md`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
for p in (ROOT, ROOT / "experiments" / "TN_ksweep"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import pandas as pd
import yaml
from src.paper_figs import layout as ly, data as dt, figures as fg, table as tb

OUT = ROOT / "output"
PUB = ROOT / "results" / "public"
ly.paper_style(8, "en")
dt.set_root(ROOT)
print("repository root:", ROOT)

## Part 1 · Table 1

Gain in test BA over *frozen*, mean over the 15 datasets (seeds 3–17), each averaged over $K \in \{2, 6, 12\}$ and
five repetitions. Source: `results/public/2026-10-20_SG9_seeds3-17/tabelle1.csv` (means) and `tabelle1_seeds.csv`
(per-dataset values for the rows *sigmoid wins*). The generated LaTeX additionally contains the rows
*datasets > frozen*, which the paper omits for space.

In [ ]:
tex, df = tb.tabelle1_latex(seeds="3–17", lang="en", out=OUT, name="table1", wins=True, compact=True)
print(tex)

# Check against the values printed in the paper
PAPER = {("G", "plus_id_end"): -0.012, ("G", "plus_id_best_val"): 0.036, ("G", "plus_sig_end"): 0.053,
         ("G", "plus_orakel"): 0.031, ("CW", "plus_id_end"): 0.045, ("CW", "plus_id_best_val"): 0.068,
         ("CW", "plus_sig_end"): 0.073, ("CW", "plus_orakel"): 0.068}
WINS = {("G", c): "15/15" for c in ("plus_id_end", "plus_id_best_val", "plus_sig_end", "plus_orakel")}
WINS.update({("CW", "plus_id_end"): "15/15", ("CW", "plus_id_best_val"): "11/15", ("CW", "plus_sig_end"): "15/15",
             ("CW", "plus_orakel"): "12/15"})
wins = tb.sigmoid_wins("3–17")
rows = []
for (lvl, col), paper in PAPER.items():
    x = float(df[df.level == lvl][col].iloc[0])
    rows.append(dict(level=lvl, column=col, computed=round(x, 3), paper=paper, equal=round(x, 3) == paper,
                     wins=wins[(lvl, col)], wins_paper=WINS[(lvl, col)], wins_equal=wins[(lvl, col)] == WINS[(lvl, col)]))
_, base, n = tb.tabelle1_df("3–17")
check = pd.DataFrame(rows)
print(f"start BA {base['start'].mittel:.3f} (paper .695), + prototypes {base['plus_prototypen'].mittel:+.3f} (paper +.060), "
      f"n = {n} datasets")
assert check.equal.all() and check.wins_equal.all()
assert round(base["start"].mittel, 3) == 0.695 and round(base["plus_prototypen"].mittel, 3) == 0.060
check

## Part 1 · Fig. 1 (decomposition over $K$)

Mean over datasets 3, 4, 5 (chosen in advance for the sweep $K = 2, \dots, 20$), bars relative to the start.
Source: `results/public/2026-10-20_SG9_seeds3-17/kverlauf_s3-5.csv`.

In [ ]:
fig1 = fg.fig_zerlegung_k(look="v1", height_cm=4.2)
_ = ly.save(fig1, "fig1", out=OUT)

## Part 1 · Fig. 2 (weight path of the example run)

Dataset 3, $K = 6$, repetition 0, $\eta = 0.003$; identity and sigmoid ($\beta = 10$) for G and CW. Source: epoch
logs and snapshots in `results/public/2026-10-20_fig2_example_run_ds3/`. The training BA is recomputed from the
snapshots; the code checks it against the logged validation BA and cost (abort if the deviation exceeds 1e-10).

In [ ]:
fig2 = fg.fig_pfad2(look="v1", height_cm=4.5)
_ = ly.save(fig2, "fig2", out=OUT)

## Part 1 · Numbers in the text

All values are read from the CSV files; `docs/numbers.md` lists file, column and row for each number.

In [ ]:
t1 = pd.read_csv(PUB / "2026-10-20_SG9_seeds3-17" / "tabelle1.csv")
t1 = t1[t1.seeds == "3–17"].set_index(["level", "groesse"])
ts = pd.read_csv(PUB / "2026-10-20_SG9_seeds3-17" / "tabelle1_seeds.csv")
ks = pd.read_csv(PUB / "2026-10-20_SG7_seeds3-15" / "kennzahlen_seeds.csv")
z = pd.read_csv(PUB / "2026-10-29_M_textzahlen_fig2" / "zahlen.csv")
kb = pd.read_csv(PUB / "2026-10-29_M_textzahlen_fig2" / "sg9_kosten_ba_fits.csv")

def zwert(level, run, what):
    q = z[(z.ebene == level) & (z.lauf.str.contains(run, regex=False)) & (z.groesse.str.startswith(what))]
    assert len(q) == 1, (level, run, what, len(q))
    return q.wert.iloc[0]

print("4.1 Decomposition (Table 1, 15 datasets)")
for lvl in ("G", "CW"):
    for g in ("plus_id_end", "plus_id_best_val", "plus_orakel"):
        r = t1.loc[(lvl, g)]
        extra = f", >= oracle on {int(r.n_ge_orakel)}/15" if g == "plus_id_best_val" else ""
        above = f"  above frozen on {int(r.n_ueber_frozen)}/15" if g != "plus_orakel" else ""
        print(f"  {lvl:2s} {g:17s} {r.mittel:+.3f}{above}{extra}")

print("\n4.2 Example run (dataset 3, K 6, rep 0)")
print("  largest angle deviation between learning-rate paths: G", zwert("G", "16 Identitäts-Pläne", "größte Winkelabweichung zwischen den Pfaden, R ≤ 1.25"),
      "| CW", zwert("CW", "16 Identitäts-Pläne", "größte Winkelabweichung zwischen den Pfaden, R ≤ 1.25"))
for lvl, run in (("G", "G_const"), ("CW", "CW_const")):
    print(f"  {lvl}: training BA best val. / end:", zwert(lvl, f"DS 3, {run},", "Train-BA / Val-BA bei Best_val").split(" / ")[0],
          "->", zwert(lvl, f"DS 3, {run},", "Train-BA / Val-BA bei Ende").split(" / ")[0],
          "| correct -> wrong:", zwert(lvl, f"DS 3, {run},", "Trainingspunkte richtig → falsch"))
print("\n4.2 All identity runs (15 datasets x K {2, 6, 12} x 5 repetitions)")
for lvl in ("G", "CW"):
    q = kb[kb.level == lvl]
    print(f"  {lvl}: cost(end) < cost(best val.) in {int(q.kosten_sinkt.sum())}/{len(q)}, "
          f"training BA(end) < training BA(best val.) in {int(q.trainba_sinkt.sum())}/{len(q)}")

print("\n4.3 Sigmoid")
print("  final angle to oracle, G: sigmoid", zwert("G", "G_sig10_e0.003_const", "Winkel am Ende"),
      "deg, identity", zwert("G", "DS 3, G_const,", "Winkel am Ende"), "deg")
k13 = ks[(ks.plan == "e0.003_const") & ks.seed.between(3, 15)]
for lvl in ("CW", "G"):
    q = k13[k13.level == lvl]
    print(f"  corner at end of training, 13 datasets, {lvl}: identity {q.ecke_echt_id.mean():.0%}, sigmoid {q.ecke_echt.mean():.0%}")
for lvl in ("G", "CW"):
    print(f"  {lvl} sigmoid end vs. frozen {t1.loc[(lvl, 'plus_sig_end')].mittel:+.3f}")
s3 = ts[(ts.seed == 3) & (ts.level == "CW")].iloc[0]
print(f"  example dataset 3, CW: sigmoid end - identity best val. = {s3.sig_end - s3.id_best_val:+.3f}")

### Expectations E1–E5 (fixed before running datasets 9–15)

Wording and thresholds: `experiments/TN_ksweep/predictions_sg7.yaml` (unchanged since it was fixed). Verdict on
datasets 9–15: `results/public/2026-10-20_SG7_seeds3-15/bewertung_sg7.csv`; per dataset 9–17:
`results/public/2026-10-20_SG9_seeds3-17/e1e5_seeds9-17.csv` (16 and 17 descriptive only).

In [ ]:
pred = yaml.safe_load((ROOT / "experiments" / "TN_ksweep" / "predictions_sg7.yaml").read_text(encoding="utf-8"))
for name, e in pred["erwartungen"].items():
    print(f"{name}: {e['groesse']} | plans {e['plaene']} | level {e['ebene']} | condition {e['erfuellt_wenn']}")
print("rule:", pred["bewertung"]["gestuetzt"])
display(pd.read_csv(PUB / "2026-10-20_SG7_seeds3-15" / "bewertung_sg7.csv"))
pd.read_csv(PUB / "2026-10-20_SG9_seeds3-17" / "e1e5_seeds9-17.csv")[["seed", "rolle", "E1", "E2", "E3", "E4", "E5"]]

## Part 2 · Re-fit a single run

One fit = one dataset (generator seed), one $K$, one repetition (seed of the initial prototypes = `base_seed + rep`),
one variant. Here: dataset 3, $K = 2$, repetition 0, G with sigmoid $\beta = 10$ and $\eta = 0.003$ (about 10–20 s).
The result is compared with the stored row from the original run (`reference_fit.csv`).

In [ ]:
import time
import run_tn as tn

cfg = tn.load_config(ROOT / "experiments" / "TN_ksweep" / "config_sig_explore.yaml")   # dataset (seed) 3
t0 = time.time()
D = tn.build_data(cfg)                       # generator, 600 train / 600 validation / 3000 test points
row, traj, hist = tn.run_fit("G_sig10_e0.003_const", K=2, rep=0, cfg=cfg, D=D)
print(f"fit and epoch log: {time.time() - t0:.1f} s, cache signature {tn.cache_signature(cfg)}")

ref = pd.read_csv(PUB / "2026-10-20_SG_explore_H0" / "reference_fit.csv").iloc[0]
cols = ["ba_init", "ba_end", "ba_best_val", "val_end", "ep_best_val", "ba_train_end"]
cmp = pd.DataFrame(dict(refit=[row[c] for c in cols], stored=[ref[c] for c in cols]), index=cols)
cmp["equal_3_digits"] = cmp.refit.round(3) == cmp.stored.round(3)
cmp

## Part 2 · Full re-run (not executed here)

Run from the repository root. Each call writes the fits to `cache/toy_H0/<signature>/` and a summary with
`fits.csv` to `results/public/<run-id>/`; runs that share a cache signature (dataset 3: TN, LR, SG explore, SG9 K
sweep) reuse each other's fits. Use exactly these run ids, the evaluation scripts read them.
Rough cost on 4 cores (`--jobs 4`): about 18 h in total, about 70 CPU hours.

```bash
python experiments/TN_ksweep/run_tn.py --config experiments/TN_ksweep/config_full.yaml       --run-id 2026-10-15_TN_ksweep_H0  --jobs 4   # ~2.1 h
python experiments/TN_ksweep/run_tn.py --config experiments/TN_ksweep/config_lr_explore.yaml --run-id 2026-10-15_LR_explore_H0 --jobs 4   # ~1.7 h
python experiments/TN_ksweep/run_tn.py --config experiments/TN_ksweep/config_sig_explore.yaml --run-id 2026-10-20_SG_explore_H0 --jobs 4  # ~0.9 h
python experiments/TN_ksweep/run_tn.py --config experiments/TN_ksweep/config_sig_confirm_s4.yaml --run-id 2026-10-20_SG_confirm_H0s4 --jobs 4 --skip-criteria-lock  # ~0.7 h
# datasets 5–17, ~0.65 h each
python experiments/TN_ksweep/run_tn.py --config experiments/TN_ksweep/config_sig_s5.yaml --run-id 2026-10-20_SG_seed5_H0 --jobs 4
#   ... same for s6 ... s17 (run id 2026-10-20_SG_seed<s>_H0)
python experiments/TN_ksweep/run_tn.py --config experiments/TN_ksweep/config_sg9_k_s3.yaml --run-id 2026-10-20_SG9_k_H0s3 --jobs 4   # ~0.6 h
python experiments/TN_ksweep/run_tn.py --config experiments/TN_ksweep/config_sg9_k_s4.yaml --run-id 2026-10-20_SG9_k_H0s4 --jobs 4   # ~1.8 h
python experiments/TN_ksweep/run_tn.py --config experiments/TN_ksweep/config_sg9_k_s5.yaml --run-id 2026-10-20_SG9_k_H0s5 --jobs 4   # ~1.8 h
# evaluation (overwrites the aggregated CSVs in results/public/)
python experiments/TN_ksweep/eval_seeds.py           # -> 2026-10-20_SG7_seeds3-15/
python experiments/TN_ksweep/eval_sg9.py             # -> 2026-10-20_SG9_seeds3-17/
python experiments/TN_ksweep/m_textzahlen_fig2.py    # -> 2026-10-29_M_textzahlen_fig2/
```